In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


# I will try training using SKLEARN as I am more familiar with it

In [2]:
import kagglehub
path = kagglehub.competition_download('titanic')

print(path)

/kaggle/input/competitions/titanic


# Lemme try Importing the data then use onehot encodder for multiple columns


In [3]:
import pandas as pd
train_data =  pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
# print(train_data.head(5))

print(train_data.isnull().sum())
# eliminating cabin column since it has high number of  na and filling age with the mean I will avoid IMPUTTER for simplicity
# Lmme drop Names it is actually useless since we know gender
train_data = train_data.drop('Cabin',axis = 1)
train_data = train_data.drop('Name', axis = 1)
train_data['Age'] = train_data['Age'].fillna(train_data['Age'].mean())
train_data = train_data.dropna(subset = ['Embarked'], axis = 0)

print(train_data.isnull().sum())

print(train_data)

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64
PassengerId    0
Survived       0
Pclass         0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Embarked       0
dtype: int64
     PassengerId  Survived  Pclass     Sex        Age  SibSp  Parch  \
0              1         0       3    male  22.000000      1      0   
1              2         1       1  female  38.000000      1      0   
2              3         1       3  female  26.000000      0      0   
3              4         1       1  female  35.000000      1      0   
4              5         0       3    male  35.000000      0      0   
..           ...       ...     ...     ...        ...    ...    ...   
886          887         0       2    male  27.000000      0      0   
887     

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_data = ['Sex', 'SibSp', 'Ticket', 'Fare', 'Embarked']
Num_data = ['PassengerId','Pclass','Age', 'Parch' ]

preprocr = ColumnTransformer(transformers = [('cat', OneHotEncoder(handle_unknown  = 'ignore',sparse_output = False),categorical_data ), ('num', 'passthrough',Num_data )])
X =  train_data.drop('Survived', axis = 1)

Y = train_data[['Survived']]

X = preprocr.fit_transform(X)
# Lemme convert it back to dataframe but not must siince I will still reconvert But I woul like to confrim something

X_feature_name =  preprocr.get_feature_names_out()

# X = pd.DataFrame(X.toarray(), columns = X_feature_name)

X = pd.DataFrame(X, columns = X_feature_name)


#print(X[:5])


print(X)



     cat__Sex_female  cat__Sex_male  cat__SibSp_0  cat__SibSp_1  cat__SibSp_2  \
0                0.0            1.0           0.0           1.0           0.0   
1                1.0            0.0           0.0           1.0           0.0   
2                1.0            0.0           1.0           0.0           0.0   
3                1.0            0.0           0.0           1.0           0.0   
4                0.0            1.0           1.0           0.0           0.0   
..               ...            ...           ...           ...           ...   
884              0.0            1.0           1.0           0.0           0.0   
885              1.0            0.0           1.0           0.0           0.0   
886              1.0            0.0           0.0           1.0           0.0   
887              0.0            1.0           1.0           0.0           0.0   
888              0.0            1.0           1.0           0.0           0.0   

     cat__SibSp_3  cat__Sib

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

#X = X
Y = Y.values.ravel()
x_train, x_test, y_train, y_test =  train_test_split(X, Y, test_size = 0.3, stratify = Y, random_state = 2)

Fdher = make_pipeline(StandardScaler(), PCA(n_components = 2), LogisticRegression(penalty = 'l2', max_iter = 10000))

Fdher.fit(x_train, y_train)
y_pred = Fdher.predict(x_test)
test_accuracy = Fdher.score(x_test,y_test)
print(f"The accuravy score is {test_accuracy}")



The accuravy score is 0.700374531835206


# The accuracy is quiet low lemme try validation like K fold and use Grid search or Halving randomsearch CV whichever gives highest


In [6]:
from sklearn.model_selection import cross_val_score
import numpy as np


npe = cross_val_score(estimator = Fdher, X = x_train, y= y_train, cv = 10, n_jobs = -1)

print(f"The accuracy score in all cv is {npe}")

print(f"The score mean is {np.mean(npe)} and std score {np.std(npe)}")



The accuracy score in all cv is [0.71428571 0.63492063 0.69354839 0.69354839 0.77419355 0.74193548
 0.74193548 0.69354839 0.74193548 0.79032258]
The score mean is 0.7220174091141833 and std score 0.04319168658096028


# Using Grid Search

In [48]:
from sklearn.svm import SVC
import scipy.stats
from sklearn.model_selection import GridSearchCV
Fdher = make_pipeline(StandardScaler(),  SVC(random_state = 9))

#Note I ignore PCA to increase accuracy
# param_range = scipy.stats.loguniform(0.0001, 1000.0)

param_range = [0.0001,0.001,0.01,1.0,10.0,100.0,1000.0]

param_grd = [{'svc__C':param_range, 'svc__kernel': ['linear'] }, {'svc__C': param_range, 'svc__gamma': param_range, 'svc__kernel': ['rbf']}]

grd = GridSearchCV(estimator = Fdher, param_grid = param_grd, cv = 10, n_jobs = -1, refit = True, scoring = 'accuracy' )

gd = grd.fit(x_train, y_train)

print(gd.best_score_)
print(f"tHE ACCURACY SCORE iS {gd.score(x_test, y_test)}")

0.8328725038402458
tHE ACCURACY SCORE iS 0.8052434456928839


In [8]:
# Halving Random Search

In [9]:
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV

param_range = scipy.stats. loguniform(0.0001, 1000)

param_grd = [{'svc__C':param_range, 'svc__kernel': ['linear'] }, {'svc__C': param_range, 'svc__gamma': param_range, 'svc__kernel': ['rbf']}]


HRS  = HalvingRandomSearchCV(Fdher, param_distributions = param_grd  ,n_candidates = 'exhaust',resource = 'n_samples', factor = 1.5, random_state = 21,  n_jobs= -1 )

hr = HRS.fit(x_train, y_train)
print(hr.best_score_)
print(f"tHE ACCURACY SCORE iS {hr.score(x_test, y_test)}")


0.7843137254901961
tHE ACCURACY SCORE iS 0.8052434456928839


In [10]:
from sklearn.model_selection import RandomizedSearchCV


rdm = RandomizedSearchCV(estimator = Fdher, param_distributions = param_grd, scoring = 'accuracy', refit = True, n_iter = 20, cv = 10, random_state = 1, n_jobs = -1)

rs = rdm.fit(x_train, y_train)
print(rs.best_score_)
print(rs.best_params_)
print(rs.score(x_test, y_test))

0.8312852022529442
{'svc__C': np.float64(0.05971247755848463), 'svc__kernel': 'linear'}
0.8052434456928839


# Letd try confusion metrics

In [11]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, matthews_corrcoef

# Grid
Y_pred = grd.predict(x_test)
confes = confusion_matrix(y_true=y_test, y_pred=Y_pred)
print(f"The confusion matrix is:\n{confes}")

prec_val = precision_score(y_true=y_test, y_pred=Y_pred)
print(f"The precision score is {prec_val}")

recall_val = recall_score(y_true=y_test, y_pred=Y_pred)
print(f"The recall score is {recall_val}")

f1_val = f1_score(y_true=y_test, y_pred=Y_pred)
print(f"The f1 score is {f1_val}")

mcc_val = matthews_corrcoef(y_true=y_test, y_pred=Y_pred)
print(f"The Matthews correlation coefficient is {mcc_val}")

#rdm

Y_pred = HRS.predict(x_test)
confes = confusion_matrix(y_true=y_test, y_pred=Y_pred)
print(f"The confusion matrix is:\n{confes}")

prec_val = precision_score(y_true=y_test, y_pred=Y_pred)
print(f"The precision score is {prec_val}")

recall_val = recall_score(y_true=y_test, y_pred=Y_pred)
print(f"The recall score is {recall_val}")

f1_val = f1_score(y_true=y_test, y_pred=Y_pred)
print(f"The f1 score is {f1_val}")

mcc_val = matthews_corrcoef(y_true=y_test, y_pred=Y_pred)
print(f"The Matthews correlation coefficient is {mcc_val}")
Y_pred = grd.predict(x_test)
confes = confusion_matrix(y_true=y_test, y_pred=Y_pred)
print(f"The confusion matrix is:\n{confes}")

prec_val = precision_score(y_true=y_test, y_pred=Y_pred)
print(f"The precision score is {prec_val}")

recall_val = recall_score(y_true=y_test, y_pred=Y_pred)
print(f"The recall score is {recall_val}")

f1_val = f1_score(y_true=y_test, y_pred=Y_pred)
print(f"The f1 score is {f1_val}")

mcc_val = matthews_corrcoef(y_true=y_test, y_pred=Y_pred)
print(f"The Matthews correlation coefficient is {mcc_val}")

#halvingrdm




The confusion matrix is:
[[141  24]
 [ 28  74]]
The precision score is 0.7551020408163265
The recall score is 0.7254901960784313
The f1 score is 0.74
The Matthews correlation coefficient is 0.5847097761829091
The confusion matrix is:
[[141  24]
 [ 28  74]]
The precision score is 0.7551020408163265
The recall score is 0.7254901960784313
The f1 score is 0.74
The Matthews correlation coefficient is 0.5847097761829091
The confusion matrix is:
[[141  24]
 [ 28  74]]
The precision score is 0.7551020408163265
The recall score is 0.7254901960784313
The f1 score is 0.74
The Matthews correlation coefficient is 0.5847097761829091


# Calculating AUC-ROC and Identfying best algorithm
    1. Check  the class imbalance
    2. Implement the the different classification algorithms and  and calculate the ROC  AUC
            > SVC
            > Random Forest
            > knn
    3. Identify best Valuation Algorithm
    3.Combine Classifiers for Majority Voting

In [15]:
t = [(d) for d in Y if d == 0]
s =  [(d) for d in Y if d ==1]
print("Before sampling")
print(f"The length of Class 0 is {len(t)}")
print(f"The length of Class 1 is {len(s)}")

# The classs are imbalance with 
# class 0 being 549
# class 1 being 340

# The best approach is to use under sampling 

from imblearn.under_sampling import RandomUnderSampler

RdmSampler = RandomUnderSampler(random_state = 21, replacement = True)
x_unders, y_unders = RdmSampler.fit_resample(X, Y)

t = [(d) for d in y_unders if d == 0]
s =  [(d) for d in y_unders if d ==1]
print("After sampling")
print(f"The length of Class 0 is {len(t)}")
print(f"The length of Class 1 is {len(s)}")



Before sampling
The length of Class 0 is 549
The length of Class 1 is 340
After sampling
The length of Class 0 is 340
The length of Class 1 is 340


# Implement different classification algorithms

In [66]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
Fdher_Dec = DecisionTreeClassifier(criterion = "entropy", max_depth = 5, random_state = 21)

Fdher_knn = KNeighborsClassifier(n_neighbors =1 , p =2, metric = "minkowski")

#Note Fdher is the pipeline for Support Vector Machine

pipe_dec = make_pipeline(StandardScaler(),Fdher_Dec )
pipe_knn = make_pipeline(StandardScaler(), Fdher_knn)

clf_labels = ["Support Vector Machine", "Decision tree", "Knn"]

param_range = [0.0001,0.001,0.01,1.0,10.0,100.0,1000.0]
param_grid_svc = {
    "svc__C": param_range,
    "svc__kernel": ["linear", "rbf"],
    "svc__gamma": ["scale", "auto"]
}

param_grid_dec = {
    "decisiontreeclassifier__max_depth": [3, 5, 7],
    "decisiontreeclassifier__criterion": ["entropy", "gini"]
}

param_grid_knn = {
    "kneighborsclassifier__n_neighbors": [1, 3, 5],
    "kneighborsclassifier__p": [1, 2]
}

#param_range = [0.0001,0.001,0.01,1.0,10.0,100.0,1000.0]

#param_grd = [{'svc__C':param_range, 'svc__kernel': ['linear'] }, {'svc__C': param_range, 'svc__gamma': param_range, 'svc__kernel': ['rbf']}]


#Param_grids = [param_grid_svc, param_grid_dec,param_grid_knn ]

# I will use cross_val_score, radomized and grid search cv as my valuation

for clf, label, param_grd in zip([Fdher,pipe_dec,pipe_knn ], clf_labels, [param_grid_svc, param_grid_dec,param_grid_knn ]):
    score = cross_val_score(estimator = clf, X= x_train, y= y_train, cv =10, n_jobs = -1, scoring = "roc_auc")
    print(f"The cross_val_score score is {score.mean()}[{label}]")
    
    rdm = RandomizedSearchCV(estimator = clf, param_distributions = param_grd, scoring = 'roc_auc', refit = True, n_iter = 20, cv = 10, random_state = 1, n_jobs = -1)
    rs = rdm.fit(x_train, y_train)
    #print(rs.best_score_)
    print(rs.best_params_)
    print(f"The randomized search cv score is {rs.score(x_test, y_test)} for [{label}]")
    grd = GridSearchCV(estimator = clf, param_grid = param_grd, cv = 10, n_jobs = -1, refit = True, scoring = 'roc_auc' )

    gd = grd.fit(x_train, y_train)

    #print(gd.best_score_)
    print(gd.best_params_)
    print(f"The grid search score iS {gd.score(x_test, y_test)} for [{label}]")

The cross_val_score score is 0.8315839592012362[Support Vector Machine]
{'svc__kernel': 'linear', 'svc__gamma': 'auto', 'svc__C': 0.01}
The randomized search cv score is 0.851574569221628 for [Support Vector Machine]
{'svc__C': 0.01, 'svc__gamma': 'scale', 'svc__kernel': 'linear'}
The grid search score iS 0.851574569221628 for [Support Vector Machine]
The cross_val_score score is 0.8047844667409885[Decision tree]


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 6 is smaller than n_iter=20. Running 6 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


{'decisiontreeclassifier__max_depth': 3, 'decisiontreeclassifier__criterion': 'gini'}
The randomized search cv score is 0.8011883541295305 for [Decision tree]
{'decisiontreeclassifier__criterion': 'gini', 'decisiontreeclassifier__max_depth': 3}
The grid search score iS 0.8011883541295305 for [Decision tree]
The cross_val_score score is 0.5554593528134719[Knn]


/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 6 is smaller than n_iter=20. Running 6 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


{'kneighborsclassifier__p': 1, 'kneighborsclassifier__n_neighbors': 5}
The randomized search cv score is 0.7847890671420084 for [Knn]
{'kneighborsclassifier__n_neighbors': 5, 'kneighborsclassifier__p': 1}
The grid search score iS 0.7847890671420084 for [Knn]



# Combining Majority Vote Classifiers

In [59]:
from sklearn.base import BaseEstimator
from sklearn.base import ClassifierMixin
from sklearn.preprocessing import LabelEncoder
from sklearn.base import clone
from sklearn.pipeline import _name_estimators
import numpy as np
import operator

class MajorityVoteClassifier(ClassifierMixin, BaseEstimator):  # ClassifierMixin must come first for sklearn 1.6+
    def __init__(self, classifiers, vote="classlabel", weights=None):
        self.classifiers = classifiers
        self.vote = vote
        self.weights = weights

    @property
    def named_classifiers(self):
        return {key: value for key, value in _name_estimators(self.classifiers)}

    def fit(self, X, y):
        if self.vote not in ('probability', 'classlabel'):
            raise ValueError("vote must be 'probability' or 'classlabel'")
        if self.weights is not None and len(self.weights) != len(self.classifiers):
            raise ValueError("Number of classifiers and weights must be equal")
        self.labelEnc_ = LabelEncoder()
        self.labelEnc_.fit(y)
        self.classes_ = self.labelEnc_.classes_
        self.classifiers_ = []
        for clf in self.classifiers:
            fitted_clf = clone(clf).fit(X, self.labelEnc_.transform(y))
            self.classifiers_.append(fitted_clf)
        return self

    def predict(self, X):
        if self.vote == "probability":
            maj_vote = np.argmax(self.predict_proba(X), axis=1)
        else:
            predictions = np.asarray([clf.predict(X) for clf in self.classifiers_]).T
            maj_vote = np.apply_along_axis(
                lambda x: np.argmax(np.bincount(x, weights=self.weights)),
                axis=1, arr=predictions
            )
        maj_vote = self.labelEnc_.inverse_transform(maj_vote)
        return maj_vote

    def predict_proba(self, X):
        probas = np.asarray([clf.predict_proba(X) for clf in self.classifiers_])
        avg_proba = np.average(probas, axis=0, weights=self.weights)
        return avg_proba

    def get_params(self, deep=True):
        if not deep:
            return super().get_params(deep=False)
        else:
            out = super().get_params(deep=False)
            for name, step in self.named_classifiers.items():
                for key, value in step.get_params(deep=True).items():
                    out[f'{name}__{key}'] = value
            return out

    def set_params(self, **params):
        for param, value in params.items():
            if '__' in param:
                name, sub_param = param.split('__', 1)
                self.named_classifiers[name].set_params(**{sub_param: value})
            else:
                setattr(self, param, value)
        return self


In [60]:
Fdher_svc = SVC(random_state = 9, probability = True)
Fdher = make_pipeline(StandardScaler(), Fdher_svc)


mv_clf = MajorityVoteClassifier(classifiers = [Fdher,pipe_dec,pipe_knn])


#Implementig Majority Classifier
Maj_class = cross_val_score(estimator = mv_clf, X = x_train, y= y_train, cv = 10, n_jobs = -1 , scoring = "roc_auc")

print(f"the majority score is {Maj_class.mean()} [Majority score]")








the majority score is 0.7971534012008841 [Majority score]


# Combining the gridsearch valuatio with SVC and Majority Classifier 

In [73]:
param_range = [0.0001,0.001,0.01,1.0,10.0,100.0,1000.0]
param_grid_svc = {
    "pipeline-1__svc__C": param_range,
    "pipeline-1__svc__kernel": ["linear", "rbf"],
    "pipeline-1__svc__gamma": ["scale", "auto"]
}

#{'svc__C': 0.01, 'svc__gamma': 'scale', 'svc__kernel': 'linear'}

grd = GridSearchCV(estimator = mv_clf, param_grid = param_grid_svc, cv = 10, n_jobs = -1, refit = True, scoring = 'roc_auc' )

gd = grd.fit(x_train, y_train)

print(gd.best_score_)
print(f"The grid search score iS {gd.score(x_test, y_test)} for [Majority Classifier]")

0.8240120577363139
The grid search score iS 0.805704099821747 for [Majority Classifier]


In [75]:
test_data = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')
# print(test_data.isnull().sum())
test_data = test_data.drop('Cabin',axis = 1)
test_data = test_data.drop('Name', axis = 1)
test_data['Age'] = test_data['Age'].fillna(test_data['Age'].mean())
test_data['Fare'] = test_data['Fare'].fillna(test_data['Fare'].mean())
test_data = test_data.dropna(subset = ['Embarked'], axis = 0)

# print(test_data.isnull().sum())

X_test_date = preprocr.transform(test_data)
Y_pred_data = grd.predict(X_test_date)
#print(Y_pred_data)

Fina_Sub = pd.DataFrame({"PassengerId": test_data['PassengerId'], "Survived": Y_pred_data})

print(Fina_Sub)
Fina_Sub.to_csv("Submission.csv", index = False)

     PassengerId  Survived
0            892         0
1            893         0
2            894         0
3            895         0
4            896         1
..           ...       ...
413         1305         0
414         1306         1
415         1307         0
416         1308         0
417         1309         0

[418 rows x 2 columns]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
